Importing Libs

In [2]:
import pandas as pd
import os

In [3]:
path = "../data/raw/heart_disease_uci.csv"

os.path.abspath(path=path)
df_raw = pd.read_csv(path)

df = df_raw.copy()

Tratamento do Target (num)

In [16]:
# Distribuição da variável target

# Renomear variável `num` para `target`
df.rename(columns={'num': 'target'}, inplace=True)

df.target.value_counts() # Valores de 1 a 4

# Conversão para 0 ou 1
df['target'] = (df['target'] > 0)
df.target.value_counts(normalize=True) * 100 # Valores binários

target
True     55.326087
False    44.673913
Name: proportion, dtype: float64

Tratamento de Missing Values

In [5]:
# Count missing values per column
missing_per_column = df.isnull().sum()
cols_with_missing = missing_per_column[missing_per_column > 0].index

columns = df.columns
total_rows = df.id.value_counts().sum()

for col in cols_with_missing:
    print(f'Column: {col} - Data Type: {df[col].dtype}')
    
for index, counts in enumerate(missing_per_column):

    pc = counts / total_rows * 100
    
    if counts > 0:
        print(f'Col: {columns[index]} - Missing Values: {counts} - Percentage: {pc}%')


Column: trestbps - Data Type: float64
Column: chol - Data Type: float64
Column: fbs - Data Type: object
Column: restecg - Data Type: str
Column: thalch - Data Type: float64
Column: exang - Data Type: object
Column: oldpeak - Data Type: float64
Column: slope - Data Type: str
Column: ca - Data Type: float64
Column: thal - Data Type: str
Col: trestbps - Missing Values: 59 - Percentage: 6.41304347826087%
Col: chol - Missing Values: 30 - Percentage: 3.260869565217391%
Col: fbs - Missing Values: 90 - Percentage: 9.782608695652174%
Col: restecg - Missing Values: 2 - Percentage: 0.21739130434782608%
Col: thalch - Missing Values: 55 - Percentage: 5.978260869565218%
Col: exang - Missing Values: 55 - Percentage: 5.978260869565218%
Col: oldpeak - Missing Values: 62 - Percentage: 6.739130434782608%
Col: slope - Missing Values: 309 - Percentage: 33.58695652173913%
Col: ca - Missing Values: 611 - Percentage: 66.41304347826087%
Col: thal - Missing Values: 486 - Percentage: 52.826086956521735%


In [11]:
num_cols_missing = ['trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
cat_cols_missing = ['fbs', 'exang']
str_cols_missing = ['restecg', 'slope', 'thal']

# Substituindo valores em colunas numéricas = mediana
print('Colunas Numéricas')
for col in num_cols_missing:
    median = df[col].median()
    df.fillna({col: median}, inplace = True)
    print(f"{col}: imputado com mediana ({median:.2f})")

# Colunas Categóricas = Moda
print('Colunas Categróricas')

for col in cat_cols_missing:
    mode = df[col].mode()[0]
    df.fillna({col: mode}, inplace = True)
    print(f"{col}: imputado com Moda ({mode})")

for col in str_cols_missing:
    mode = df[col].mode()[0]
    df.fillna({col: mode}, inplace = True)
    print(f"{col}: imputado com Moda ({mode})")


Colunas Numéricas
trestbps: imputado com mediana (130.00)
chol: imputado com mediana (223.00)
thalch: imputado com mediana (140.00)
oldpeak: imputado com mediana (0.50)
ca: imputado com mediana (0.00)
Colunas Categróricas
fbs: imputado com Moda (False)
exang: imputado com Moda (False)
restecg: imputado com Moda (normal)
slope: imputado com Moda (flat)
thal: imputado com Moda (normal)


In [13]:
print(f"Shape após imputação: {df.shape}")
print("Valores ausentes restantes por coluna:")
print(df.isnull().sum())

Shape após imputação: (920, 16)
Valores ausentes restantes por coluna:
id          0
age         0
sex         0
dataset     0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalch      0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
num         0
dtype: int64


Encoding

In [40]:
# Classificar a coluna 'age' em categorias
bins = [0, 40, 55, 65, 80]  # Definir os intervalos de idade
labels=['young', 'mid_age', 'senior', 'elderly']


df['age_category'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)
df.drop(columns=['age'], inplace=True)

# Exibir as primeiras linhas para verificar
df.head()


,id,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,target,age_category
0,1,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,False,senior
1,2,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,True,elderly
2,3,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,True,elderly
3,4,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,False,young
4,5,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,False,mid_age


In [41]:
# Remova a coluna `dataset`
df.drop(columns=['dataset', 'id'], inplace=True)

# One-hot encoding das variáveis categóricas relevantes
categorical_cols = ['sex', 'cp', 'restecg', 'slope', 'thal', 'age_category']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(f"Shape após one-hot encoding: {df_encoded.shape}")
df_encoded.head()

Shape após one-hot encoding: (920, 21)


,trestbps,chol,fbs,thalch,exang,oldpeak,ca,target,sex_Male,cp_atypical angina,...,cp_typical angina,restecg_normal,restecg_st-t abnormality,slope_flat,slope_upsloping,thal_normal,thal_reversable defect,age_category_mid_age,age_category_senior,age_category_elderly
0,145.0,233.0,True,150.0,False,2.3,0.0,False,True,False,...,True,False,False,False,False,False,False,False,True,False
1,160.0,286.0,False,108.0,True,1.5,3.0,True,True,False,...,False,False,False,True,False,True,False,False,False,True
2,120.0,229.0,False,129.0,True,2.6,2.0,True,True,False,...,False,False,False,True,False,False,True,False,False,True
3,130.0,250.0,False,187.0,False,3.5,0.0,False,True,False,...,False,True,False,False,False,True,False,False,False,False
4,130.0,204.0,False,172.0,False,1.4,0.0,False,False,True,...,False,False,False,False,True,True,False,True,False,False


Exportação de Dataset processado

In [42]:
file_path = r'../data/processed/heart_disease_uci_processed.csv'
df_encoded.to_csv(file_path, index=False)